# Notebook 3 — Backtest Results

Walk-forward backtest results for the top-5 selected pairs.
Honest IS vs OOS comparison with limitations discussion.

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from src.data_loader import load_prices, split_in_out_of_sample
from src.backtester import backtest_portfolio, TRADING_DAYS_PER_YEAR
from src.metrics import compute_metrics, metrics_table
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('Imports OK')

In [ ]:
prices = load_prices()
prices_is, prices_oos = split_in_out_of_sample(prices, in_sample_end='2020-12-31')
with open('../data/selected_pairs.pkl', 'rb') as f:
    selected_pairs = pickle.load(f)
print(f'Loaded {len(selected_pairs)} pairs')
for sp in selected_pairs:
    print(f'  #{sp.rank} {sp.analysis.ticker_y}/{sp.analysis.ticker_x}  HL={sp.half_life_days:.1f}d')

## 1. Run Walk-Forward Backtest

In [ ]:
INITIAL_CAPITAL = 1_000_000
hl_map = {f"{sp.analysis.ticker_y}~{sp.analysis.ticker_x}": sp.half_life_days for sp in selected_pairs}

print('Running in-sample backtest (2015-2020)...')
port_is = backtest_portfolio(selected_pairs, prices_is, hl_map, total_capital=INITIAL_CAPITAL)
print(f'  IS trades: {sum(r.n_trades for r in port_is.pair_results)}')

print('Running out-of-sample backtest (2021-present)...')
port_oos = backtest_portfolio(selected_pairs, prices_oos, hl_map, total_capital=INITIAL_CAPITAL)
print(f'  OOS trades: {sum(r.n_trades for r in port_oos.pair_results)}')
print('Done.')

## 2. Metrics: In-Sample vs Out-of-Sample

In [ ]:
is_trades = [t for r in port_is.pair_results for t in r.trades]
oos_trades = [t for r in port_oos.pair_results for t in r.trades]
m_is = compute_metrics(port_is.daily_pnl, trades=is_trades, period_label='In-Sample 2015-2020')
m_oos = compute_metrics(port_oos.daily_pnl, trades=oos_trades, period_label='Out-of-Sample 2021+')
print(metrics_table(m_is, m_oos).to_string())

## 3. Equity Curves

In [ ]:
equity_is = INITIAL_CAPITAL + port_is.equity_curve
equity_oos = INITIAL_CAPITAL + port_oos.equity_curve

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(equity_is.index, equity_is.values, color='steelblue', lw=1.5, label='Strategy (IS)')
axes[0].axhline(INITIAL_CAPITAL, color='black', lw=0.8, ls=':', alpha=0.5, label='Starting capital')
axes[0].set_title(f'In-Sample (2015-2020)  |  Sharpe={m_is.sharpe_ratio:.2f}')
axes[0].set_ylabel('Portfolio Value (AUD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(equity_oos.index, equity_oos.values, color='darkorange', lw=1.5, label='Strategy (OOS)')
axes[1].axhline(INITIAL_CAPITAL, color='black', lw=0.8, ls=':', alpha=0.5, label='Starting capital')
axes[1].set_title(f'Out-of-Sample (2021+)  |  Sharpe={m_oos.sharpe_ratio:.2f}')
axes[1].set_ylabel('Portfolio Value (AUD)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Walk-Forward Strategy Equity Curves', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Drawdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (equity, label, color) in zip(axes, [
    (equity_is, 'In-Sample', 'steelblue'),
    (equity_oos, 'Out-of-Sample', 'darkorange'),
]):
    if len(equity) == 0: continue
    dd = equity - equity.cummax()
    ax.fill_between(dd.index, dd.values, 0, color=color, alpha=0.35)
    ax.plot(dd.index, dd.values, color=color, lw=0.9)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{label} Drawdown  |  Max=${dd.min():,.0f}')
    ax.set_ylabel('Drawdown (AUD)')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Rolling 6-Month Sharpe

In [ ]:
ROLL = 126
full_pnl = pd.concat([port_is.daily_pnl, port_oos.daily_pnl]).sort_index()
r_mean = full_pnl.rolling(ROLL, min_periods=ROLL//2).mean()
r_std = full_pnl.rolling(ROLL, min_periods=ROLL//2).std(ddof=1)
r_sharpe = (r_mean / r_std) * np.sqrt(TRADING_DAYS_PER_YEAR)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(r_sharpe.index, r_sharpe.values, color='teal', lw=1.2)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.axhline(1, color='green', lw=0.7, ls=':', alpha=0.7, label='Sharpe=1')
if len(port_oos.daily_pnl) > 0:
    ax.axvline(port_oos.daily_pnl.index[0], color='red', lw=1.2, ls=':', label='IS/OOS split')
ax.set_title('Rolling 6-Month Sharpe Ratio')
ax.set_ylabel('Sharpe')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Per-Pair Contribution

In [ ]:
print('=== In-Sample Per-Pair Contribution ===')
is_pair_pnls = {}
for r in port_is.pair_results:
    k = f'{r.ticker_y}/{r.ticker_x}'
    is_pair_pnls[k] = r.daily_pnl
    print(f'  {k:<25} ${r.total_net_pnl:>10,.0f}   {r.n_trades} trades')

print()
print('=== Out-of-Sample Per-Pair Contribution ===')
oos_pair_pnls = {}
for r in port_oos.pair_results:
    k = f'{r.ticker_y}/{r.ticker_x}'
    oos_pair_pnls[k] = r.daily_pnl
    print(f'  {k:<25} ${r.total_net_pnl:>10,.0f}   {r.n_trades} trades')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (d, title) in zip(axes, [(is_pair_pnls, 'In-Sample'), (oos_pair_pnls, 'Out-of-Sample')]):
    labels = list(d.keys())
    totals = [s.sum() for s in d.values()]
    colors = ['steelblue' if t >= 0 else 'crimson' for t in totals]
    ax.bar(labels, totals, color=colors, edgecolor='black', lw=0.5)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{title} — Per-Pair PnL')
    ax.set_ylabel('Net PnL (AUD)')
    ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=8)
    ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. IS vs OOS Degradation Discussion

In [ ]:
print('IS vs OOS PERFORMANCE SUMMARY')
print('=' * 50)
rows = [
    ('Sharpe', m_is.sharpe_ratio, m_oos.sharpe_ratio),
    ('Ann Return ($)', m_is.annualised_return, m_oos.annualised_return),
    ('Max DD ($)', m_is.max_drawdown, m_oos.max_drawdown),
    ('Win Rate', m_is.win_rate, m_oos.win_rate),
    ('Profit Factor', m_is.profit_factor, m_oos.profit_factor),
]
for name, iv, ov in rows:
    ratio = f'{ov/iv:.2f}x' if abs(iv) > 1e-8 else 'N/A'
    print(f'  {name:<20} IS={iv:>10.3f}  OOS={ov:>10.3f}  Ratio={ratio}')

print()
if m_oos.sharpe_ratio < 0:
    print('HONEST RESULT: OOS Sharpe is negative.')
    print('Possible causes: overfitting, regime change, survivorship bias, cost drag.')
elif m_oos.sharpe_ratio < m_is.sharpe_ratio * 0.5:
    print('HONEST RESULT: Significant OOS degradation (>50% Sharpe decay).')
    print('IS performance reflects in-sample fit; OOS is the true evaluation.')
else:
    print('HONEST RESULT: Strategy retained reasonable OOS performance.')
    print('Still, IS results overstate expected live performance due to lookahead in pair selection.')

In [ ]:
print()
print('=' * 60)
print('KNOWN LIMITATIONS')
print('=' * 60)
lims = [
    '1. SURVIVORSHIP BIAS: Universe is ASX 50 as of 2024. Delisted stocks absent.',
    '2. NO INTRADAY MODEL: Assumes daily close-to-close fills.',
    '3. BORROW ASSUMED: 50 bps borrow may underestimate real costs for illiquid names.',
    '4. MARKET IMPACT: Position sizing ignores ASX daily volume constraints.',
    '5. STATIC PAIRS: Re-selection would be needed in a live implementation.',
]
for l in lims:
    print(f'  {l}')
